# 02 – Kontrollierte Klassifikationsdaten
Mit synthetischen Daten kennen wir den Erzeugungsprozess und können Klassen, Rauschen und Trennbarkeit gezielt einstellen. Wir visualisieren zuerst, teilen sauber auf und trainieren anschließend eine Pipeline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score, classification_report

## Daten gezielt erzeugen
Es gibt zwei numerische Merkmale und eine seltenere positive Klasse. random_state macht das Beispiel reproduzierbar.

In [ ]:
X_array, y_array = make_classification(n_samples=600, n_features=2, n_informative=2, n_redundant=0, n_clusters_per_class=1, weights=[0.8, 0.2], class_sep=1.4, flip_y=0.03, random_state=42)
X = pd.DataFrame(X_array, columns=["Merkmal 1", "Merkmal 2"])
y = pd.Series(y_array, name="Klasse")
print(y.value_counts().rename(index={0:"Klasse 0",1:"Klasse 1"}))

In [ ]:
fig, ax=plt.subplots(figsize=(8,6))
for klasse,farbe in [(0,"steelblue"),(1,"firebrick")]:
    maske=y==klasse
    ax.scatter(X.loc[maske,"Merkmal 1"],X.loc[maske,"Merkmal 2"],label=f"Klasse {klasse}",alpha=.65,color=farbe)
ax.set(title="Kontrollierte Klassifikationsdaten",xlabel="Merkmal 1",ylabel="Merkmal 2"); ax.legend(); ax.grid(alpha=.2); plt.show()

## Erst splitten, dann lernen
stratify erhält die Klassenanteile. Der Testdatensatz bleibt bis zur Bewertung unangetastet.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
print("Training [%]:\n",(y_train.value_counts(normalize=True)*100).round(1))
print("Test [%]:\n",(y_test.value_counts(normalize=True)*100).round(1))

## Pipeline: Skalierung und Modell
fit lernt zuerst Mittelwert und Standardabweichung aus X_train und danach das Modell. Testdaten fließen nicht in diese Parameter ein.

In [ ]:
pipeline=make_pipeline(StandardScaler(),LogisticRegression())
pipeline.fit(X_train,y_train)
y_pred=pipeline.predict(X_test)
print("Accuracy:",round(accuracy_score(y_test,y_pred),3))
print("Balanced Accuracy:",round(balanced_accuracy_score(y_test,y_pred),3))
print(classification_report(y_test,y_pred))
ConfusionMatrixDisplay.from_predictions(y_test,y_pred,cmap="Blues"); plt.show()

## Entscheidungsbereiche sichtbar machen

In [ ]:
x1=np.linspace(X["Merkmal 1"].min()-.5,X["Merkmal 1"].max()+.5,250)
x2=np.linspace(X["Merkmal 2"].min()-.5,X["Merkmal 2"].max()+.5,250)
xx,yy=np.meshgrid(x1,x2)
raster=pd.DataFrame({"Merkmal 1":xx.ravel(),"Merkmal 2":yy.ravel()})
zz=pipeline.predict_proba(raster)[:,1].reshape(xx.shape)
fig,ax=plt.subplots(figsize=(8,6)); ax.contourf(xx,yy,zz,levels=np.linspace(0,1,11),cmap="RdBu_r",alpha=.35); ax.contour(xx,yy,zz,levels=[.5],colors="black")
ax.scatter(X_test["Merkmal 1"],X_test["Merkmal 2"],c=y_test,cmap="bwr",edgecolor="white"); ax.set(title="Testdaten und Modellentscheidung",xlabel="Merkmal 1",ylabel="Merkmal 2"); plt.show()

Die Grafik hilft beim Verständnis, ersetzt aber keine Metrik. In realen Datensätzen mit vielen Merkmalen lässt sich die Entscheidungsfläche nicht direkt zeichnen.